# Graph Neural Networks for Proteins — Tutorial

This notebook walks through GNN concepts using protein data, mirroring the standalone scripts `gnn_l1` … `gnn_l5`. We cover:
- Representing a protein as a graph (PyTorch Geometric `Data` object)
- Visualising the resulting graph
- Generating a synthetic per-residue prediction task
- Training a simple **GCN** and comparing it to a per-node **MLP baseline**
- Pointers to graph-level prediction, pLM + GNN, and equivariant models

## 1. Install and Import Libraries

If you haven't already, install the requirements: `pip install -r requirements.txt`.

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
import torch_geometric
import networkx as nx
import matplotlib.pyplot as plt

print(f"PyTorch:           {torch.__version__}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")

## 2. Represent a Protein as a Graph

A protein is a chain of amino acids. We can represent it as a graph where:

- **Nodes** = residues (amino acids)
- **Node features** = the amino acid identity (one-hot, 20 dims)
- **Edges** = sequential or spatial proximity

Below, we build a **sequence graph** — each residue is connected to its `WINDOW` nearest neighbours in the sequence.

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {a: i for i, a in enumerate(AMINO_ACIDS)}

# Edit this string to try different proteins
SEQUENCE = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
WINDOW = 2  # connect each residue to its WINDOW nearest neighbours in sequence

def one_hot(seq):
    x = torch.zeros(len(seq), 20)
    for i, a in enumerate(seq):
        if a in AA_TO_IDX:
            x[i, AA_TO_IDX[a]] = 1.0
    return x

def sequence_graph(seq, window):
    n = len(seq)
    edges = []
    for i in range(n):
        for d in range(1, window + 1):
            if i + d < n:
                edges.append((i, i + d))
                edges.append((i + d, i))
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=one_hot(seq), edge_index=edge_index)

g = sequence_graph(SEQUENCE, WINDOW)
print(f"Sequence length:  {len(SEQUENCE)}")
print(f"Number of nodes:  {g.num_nodes}")
print(f"Number of edges:  {g.num_edges}")
print(f"x shape:          {tuple(g.x.shape)}")
print(f"edge_index shape: {tuple(g.edge_index.shape)}")

In [ ]:
# Visualise the graph using networkx + matplotlib
G = nx.Graph()
for i in range(g.num_nodes):
    G.add_node(i)
for s, t in g.edge_index.t().tolist():
    G.add_edge(s, t)

plt.figure(figsize=(8, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_size=80, with_labels=False, node_color="skyblue", edge_color="lightgrey")
plt.title(f"Sequence graph (window={WINDOW})")
plt.show()

## 3. Generate a Synthetic Node-Classification Dataset

To train a GNN, we need many graphs with labels. We generate random protein-like sequences with this synthetic per-residue label:

> A residue is labelled `1` if it is hydrophobic **AND** most of its sequence neighbours are also hydrophobic.

This task is **context-dependent**: a per-residue model that ignores neighbours can't solve it well. A GNN can. (See `gnn_l2_node_classification.py`.)

In [ ]:
HYDROPHOBIC = set("AVLIFMWY")

def label_residues(seq, window=3, threshold=0.6):
    n = len(seq)
    labels = []
    for i in range(n):
        if seq[i] not in HYDROPHOBIC:
            labels.append(0)
            continue
        lo, hi = max(0, i - window), min(n, i + window + 1)
        nbrs = seq[lo:i] + seq[i + 1 : hi]
        if not nbrs:
            labels.append(0)
            continue
        frac = sum(1 for a in nbrs if a in HYDROPHOBIC) / len(nbrs)
        labels.append(1 if frac >= threshold else 0)
    return torch.tensor(labels, dtype=torch.long)

def make_dataset(n_graphs, seed=0):
    rng = random.Random(seed)
    data = []
    for _ in range(n_graphs):
        L = rng.randint(40, 120)
        seq = "".join(rng.choice(AMINO_ACIDS) for _ in range(L))
        data.append(
            Data(
                x=one_hot(seq),
                edge_index=sequence_graph(seq, 3).edge_index,
                y=label_residues(seq),
            )
        )
    return data

train_set = make_dataset(200, seed=0)
test_set = make_dataset(50, seed=1)

n_pos = sum((d.y == 1).sum().item() for d in train_set)
n_total = sum(d.y.numel() for d in train_set)
print(f"Train graphs: {len(train_set)}, test graphs: {len(test_set)}")
print(f"Positive label fraction: {n_pos / n_total:.3f}")
print(f"Always-predict-majority accuracy: {max(n_pos / n_total, 1 - n_pos / n_total):.3f}")

## 4. Train a GCN vs an MLP Baseline

Both models have the same number of layers and the same hidden size. The **only** difference is whether they use the graph's edges:

- **MLP**: each node's prediction depends only on its own features.
- **GCN**: each node's prediction is informed by its 2-hop neighbourhood via message passing.

If the GCN beats the MLP, that's the value of the graph structure on this task.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_c, hidden, out_c):
        super().__init__()
        self.conv1 = GCNConv(in_c, hidden)
        self.conv2 = GCNConv(hidden, out_c)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        return self.conv2(h, edge_index)


class MLP(torch.nn.Module):
    def __init__(self, in_c, hidden, out_c):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_c, hidden)
        self.fc2 = torch.nn.Linear(hidden, out_c)

    def forward(self, x, edge_index):  # edge_index unused on purpose
        return self.fc2(F.relu(self.fc1(x)))


def train_one(model, loader, opt, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(batch.x, batch.edge_index), batch.y)
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index).argmax(dim=-1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.numel()
    return correct / total


device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8)

results = {}
for name, ModelCls in [("MLP (no edges)", MLP), ("GCN (uses edges)", GCN)]:
    model = ModelCls(20, 32, 2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    print(f"\nTraining {name}...")
    for ep in range(20):
        loss = train_one(model, train_loader, opt, device)
        if (ep + 1) % 5 == 0:
            acc = evaluate(model, test_loader, device)
            print(f"  epoch {ep + 1:3d}  loss={loss:.3f}  test_acc={acc:.3f}")
    results[name] = evaluate(model, test_loader, device)

print("\nFinal results:")
for k, v in results.items():
    print(f"  {k:20s}  {v:.3f}")

## 5. Where to Go Next

You've trained a basic GCN on a per-residue task. The standalone scripts in this folder go further:

| Script | What it adds |
|---|---|
| `gnn_l3_graph_classification.py` | Whole-protein prediction via **GAT + global pooling**, trained on real DeepSol solubility data. |
| `gnn_l4_plm_plus_gnn.py` | The bridge to the pLM track: use **ESM-2 embeddings** as node features instead of one-hot AA. One of the most reliably effective patterns in protein ML. |
| `gnn_l5_equivariant_gnn.py` | Build a minimal **EGNN** from scratch. Numerically verify rotation/translation equivariance. Foundation for force prediction, structure denoising, ProteinMPNN-style design. |

Run them directly from the terminal:

```bash
python gnn_l3_graph_classification.py
python gnn_l4_plm_plus_gnn.py
python gnn_l5_equivariant_gnn.py
```